In [8]:
print("Radha!")

Radha!


In [9]:
from typing import TypedDict, List, Annotated
import operator
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import  load_dotenv


In [10]:
# ==================================================================
# 1. Define State
# ==================================================================
class AssistantState(TypedDict):
    messages: Annotated[List[BaseMessage], operator.add]


In [11]:
# ==================================================================
# 2. Initialize Model & Nodes
# ==================================================================
load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def chat_node(state: AssistantState) -> dict:
    """Node that processes conversation history and invokes the LLM."""
    response = llm.invoke(state["messages"])
    return {"messages": [response]}


In [12]:
# ==================================================================
# 3. Build & Compile Graph with Checkpointer
# ==================================================================
def build_assistant_graph():
    builder = StateGraph(AssistantState)
    builder.add_node("chat", chat_node)
    builder.add_edge(START, "chat")
    builder.add_edge("chat", END)
    
    checkpointer = InMemorySaver()
    return builder.compile(checkpointer=checkpointer)


In [13]:
# ==================================================================
# 4. Helper Function to Interact with Assistant
# ==================================================================
def send_message(graph, user_message: str, thread_id: str):
    """Sends a user message using a specific thread_id for state tracking."""
    config = {"configurable": {"thread_id": thread_id}}
    input_state = {"messages": [HumanMessage(content=user_message)]}
    
    output = graph.invoke(input_state, config)
    last_message = output["messages"][-1]
    
    print(f"User ({thread_id}): {user_message}")
    print(f"Assistant: {last_message.content}\n")
    return output


# Initialize Graph Application
app = build_assistant_graph()

In [14]:
# ==================================================================
# 5. Execution Example: Session 1 (User A)
# ==================================================================
print("--- Session 1: User A ---")
send_message(app, "My name is Rahul and I like Python programming.", thread_id="user-101")
send_message(app, "What is my name and favorite programming language?", thread_id="user-101")

--- Session 1: User A ---
User (user-101): My name is Rahul and I like Python programming.
Assistant: Hi Rahul! It's great to hear that you enjoy Python programming. Python is a versatile language that's widely used for web development, data analysis, artificial intelligence, automation, and more. Do you have any specific projects you're working on or topics in Python that you're interested in?

User (user-101): What is my name and favorite programming language?
Assistant: Your name is Rahul, and your favorite programming language is Python.



{'messages': [HumanMessage(content='My name is Rahul and I like Python programming.', additional_kwargs={}, response_metadata={}),
  AIMessage(content="Hi Rahul! It's great to hear that you enjoy Python programming. Python is a versatile language that's widely used for web development, data analysis, artificial intelligence, automation, and more. Do you have any specific projects you're working on or topics in Python that you're interested in?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 17, 'total_tokens': 71, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_d48d865f86', 'id': 'chatcmpl-EMact6KQiGrY7ANi2b8WgJoyay1mW', 'service_tier': 'defaul

In [15]:
# ==================================================================
# 6. Execution Example: Session 2 (User B - Isolated Context)
# ==================================================================
print("--- Session 2: User B ---")
send_message(app, "What is my name?", thread_id="user-102")

--- Session 2: User B ---
User (user-102): What is my name?
Assistant: I'm sorry, but I don't have access to personal information about you unless you share it with me. How can I assist you today?



{'messages': [HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}),
  AIMessage(content="I'm sorry, but I don't have access to personal information about you unless you share it with me. How can I assist you today?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 12, 'total_tokens': 39, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_1d2403c701', 'id': 'chatcmpl-EMacxsNQWakI9tgwgpXUEcNNYleFp', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a08bda-0a35-7593-bd51-370a7cec1a50-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 12, 'output_tokens': 27, 'tota

In [16]:
# ==================================================================
# 7. Execution Example: Resume Session 1 (User A - Memory Restored)
# ==================================================================
print("--- Resume Session 1: User A ---")
send_message(app, "Do you remember what language I like?", thread_id="user-101")

--- Resume Session 1: User A ---
User (user-101): Do you remember what language I like?
Assistant: Yes, you mentioned that you like Python programming.



{'messages': [HumanMessage(content='My name is Rahul and I like Python programming.', additional_kwargs={}, response_metadata={}),
  AIMessage(content="Hi Rahul! It's great to hear that you enjoy Python programming. Python is a versatile language that's widely used for web development, data analysis, artificial intelligence, automation, and more. Do you have any specific projects you're working on or topics in Python that you're interested in?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 17, 'total_tokens': 71, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_d48d865f86', 'id': 'chatcmpl-EMact6KQiGrY7ANi2b8WgJoyay1mW', 'service_tier': 'defaul